In [ ]:
#Notes from 28th september 2026
# Do not quantize during finetuning
# log eval loss

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip install -q \
    "transformers==5.18.0" \
    "trl==1.14.2" \
    "peft==0.21.2" \
    "accelerate==1.15.0" \
    "datasets==5.0.1" \
    "bitsandbytes==0.50.2" \
    "wandb==0.28.1"

In [ ]:
!pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.4 MB/s eta 0:00:00


In [ ]:
import torchao
print(torchao.__version__)

0.16.0


In [ ]:
!pip install weave

In [ ]:
from google.colab import userdata
import torch
import os

hf_token = userdata.get("hf_write_token")
WANDB_PROJECT_NAME = 'AgriFineTuningDemo-Sep2026'
# print(hf_token)

In [ ]:
import wandb
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: debasishdhal to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
os.environ["WANDB_API_KEY"] = userdata.get("wandb")

In [ ]:
bch_dataset_id = "bharatgenai/BhashaBench-Krishi" #https://huggingface.co/datasets/bharatgenai/BhashaBench-Krishi
ft_dataset_id = "KisanVaani/agriculture-qa-english-only" #https://huggingface.co/datasets/KisanVaani/agriculture-qa-english-only
base_model_id = "meta-llama/Llama-3.2-3B-Instruct"
model_name = base_model_id.split("/")[-1]

# Loading of dataset/model

In [ ]:
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset

benchmark_dataset_raw = load_dataset(path=bch_dataset_id, name='English', split='test', token=hf_token).to_pandas()
benchmark_dataset_raw.describe()

README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

English/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.10MB            

English/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/12648 [00:00<?, ? examples/s]

,id,question,option_a,option_b,option_c,option_d,correct_answer,language,subject_domain,question_type,question_level
count,12648,12648,12648,12648,12648,12648,12648,12648,12648,12648,12648
unique,12648,12550,8124,8319,8401,8359,4,1,25,5,3
top,f18a0f65-ab27-4ffb-a109-d00889ddd462,Match List I with List II,Both Statement I and Statement II are true,Both Statement I and Statement II are false,Statement (I) is correct but Statement (II) is...,Statement (I) is incorrect but Statement (II) ...,A,en,Agronomy,MCQ,Medium
freq,1,33,173,175,154,159,3636,12648,3744,10812,5909


In [ ]:
ft_dataset_raw = load_dataset(path=ft_dataset_id, split='train').to_pandas()
ft_dataset_raw.describe()

README.md:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.97MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/22615 [00:00<?, ? examples/s]

,question,answers
count,22615,22615
unique,2225,2281
top,What is total seed population in an acre?,Cassava
freq,52,50


In [ ]:
!pip install --upgrade weave

In [ ]:
import weave
weave.init(WANDB_PROJECT_NAME)

weave: Logged in as Weights & Biases user: debasishdhal.
weave: View Weave data at https://wandb.ai/debasishdhal/AgriFineTuningDemo-Sep2026/weave


The previous `weave` upgrade pulled in `opentelemetry-sdk==1.45.0` which seems to be incompatible with the current `weave` version in how it accesses the `OTLPSpanExporter` object. To fix this, we will explicitly downgrade `opentelemetry-sdk` and `opentelemetry-api` to `1.42.1` to provide a compatible interface.

In [ ]:
# Uninstall current opentelemetry packages
# !pip uninstall -y opentelemetry-sdk opentelemetry-api--1.43.

# Install compatible versions
!pip install opentelemetry-sdk==1.42.1 opentelemetry-api==1.42.1

Now that the `opentelemetry` dependencies are at a compatible version, let's retry initializing `weave`.

In [ ]:
import weave
weave.init(WANDB_PROJECT_NAME)

AttributeError: 'OTLPSpanExporter' object has no attribute '_session'

In [ ]:
!pip show weave

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
base_model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=base_model_id, token=hf_token, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=hf_token)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
import gc
import torch

# 1. Delete model and tokenizer variables if they exist
if "model" in globals():
    del base_model
if "tokenizer" in globals():
    del tokenizer

# 2. Force Python garbage collection
gc.collect()

# 3. Clear PyTorch's internal VRAM cache
torch.cuda.empty_cache()


# Benchmarking

In [ ]:
BASE_PROMPT = """
            READ THE FOLLOWING QUESTIONS, ALONG WITH 4 OPTIONS. SELECT ONLY ONE AS THE CORRECT ANSWER
            Sample input:
            What is the capital of France?
            A: Moscow
            B: Paris
            C: London
            D: Prague

            Answer: B (Only provide the option as output, nothing else) \n\n\n
"""

question = """
          What is the capital of India
          """

options = """
          A: Dhaka
          B: Colombo
          C: Kualalampur
          D: Delhi
          """

FINAL_PROMPT = BASE_PROMPT + question + options + "What is the correct answer? Only provide the correct option (A/B/C/D)"

messages = [{"role": "user", "content": FINAL_PROMPT}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Modern practice: Explicitly configure generation settings separately
generation_config = GenerationConfig(
    max_new_tokens=100,
    do_sample=True,
    temperature=0.00001,
    # top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)

with torch.no_grad():
    output_ids = base_model.generate(
        **inputs,
        generation_config=generation_config # Modern clean syntax passing the config instance
    )

# Slicing the output matrix dynamically to only take text generated *after* the input text lengths
generated_tokens = output_ids[0, inputs.input_ids.shape[-1]:]
print(tokenizer.decode(generated_tokens, skip_special_tokens=True))


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


D


In [ ]:
def build_prompt_string(row):
    question_block = f"{row['question']}\n"
    options_block = f"A: {row['option_a']}\nB: {row['option_b']}\nC: {row['option_c']}\nD: {row['option_d']}\n"

    final_user_content = (
        BASE_PROMPT +
        question_block +
        options_block +
        "What is the correct answer? Only provide the correct option (A/B/C/D)"
    )

    # Apply modern chat template formatting natively matching model requirements
    messages = [{"role": "user", "content": final_user_content}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 4. Manual Loop Benchmarking Function
import pandas as pd
import tqdm
import wandb

def run_benchmark(df: pd.DataFrame, model, index, generation_config, collection_name="BenchmarkResults") -> pd.DataFrame:
    run = wandb.init(
        project=WANDB_PROJECT_NAME,
        name=f"run-{model.config._name_or_path.split('/')[-1]}-top-{index}",
        config={
            "model_name": model.config._name_or_path,
            "evaluation_samples": index,
            "max_new_tokens": generation_config.max_new_tokens,
            "do_sample": generation_config.do_sample,
            "temperature": generation_config.temperature
        }
    )
    client = weaviate.connect_to_local()

    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)

    model_answers = []
    matches = []
    df = df.iloc[0:index]
    print(f"Running modern manual inference loop on {len(df)} rows...")

    # Iterating over the dataframe rows manually without high-level pipe systems
    for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
        formatted_prompt = build_prompt_string(row)

        # Tokenize and push directly to the target shard layer device
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

        # Safe evaluation block preventing tracking graph overheads
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                generation_config=generation_config
            )

        # Modern practice: Dynamically slice generation payload matching post-prompt arrays
        generated_tokens = output_ids[0, inputs.input_ids.shape[-1]:]
        raw_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        # Validation Layer: Extract just the alphabetical selection
        clean_ans = "".join(char for char in raw_text if char.isalpha()).upper()
        clean_ans = clean_ans[0] if len(clean_ans) > 0 else "INVALID"

        # Verification layer comparing predictions against grounds truth
        expected = str(row["correct_answer"]).strip().upper()
        is_match = (clean_ans == expected)

        model_answers.append(clean_ans)
        matches.append(is_match)

    # Append analytical outputs cleanly
    df_result = df.copy()
    df_result["model_predicted"] = model_answers
    df_result["is_correct"] = matches

    accuracy = (df_result["is_correct"].sum() / len(df_result)) * 100
    print(f"\n🚀 Benchmarking Complete! Final Accuracy Score: {accuracy:.2f}%")

    return df_result

run_benchmark(df=benchmark_dataset_raw, model=base_model, index=500)

Running modern manual inference loop on 500 rows...


100%|██████████| 500/500 [05:30<00:00,  1.51it/s]


🚀 Benchmarking Complete! Final Accuracy Score: 39.40%


,id,question,option_a,option_b,option_c,option_d,correct_answer,language,subject_domain,question_type,question_level,model_predicted,is_correct
0,80a5aea2-be15-4b6d-b98f-4b0eca30f50e,It is state or condition of atmosphere at give...,Climate,Weather,Environment,Atmosphere,B,en,Agronomy,MCQ,Easy,B,True
1,6e6014d1-5f74-4192-ab52-b037ca469844,It refers to the water vapour content of the a...,Vapour pressure deficit,Atmospheric pressure,Vapour pressure,Humidity,D,en,Agronomy,MCQ,Easy,C,False
2,dfdc695b-c94f-4a0d-a002-9b71e4cdbcd1,Wind velocity is measured by?,Anemometer,Barometer,Altimeter,Odometer,A,en,Agronomy,MCQ,Easy,A,True
3,0a70305f-d7a8-4723-998a-4ca2b3293967,It is an aggregate of minute drops of water su...,Water aggregation,Fog,Cloud,Droplets,C,en,Agronomy,MCQ,Easy,B,False
4,eea84ced-5c3a-4f26-87e1-4574743beb48,It is an automated method of surface irrigation.?,Surge irrigation,Corrugation,Basin method,Cablegation,D,en,Agronomy,MCQ,Medium,A,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,49b14875-1296-4b37-adb1-1052cf87319a,Somatic hybridization is achieved through?,Grafting,Consugation,Protoplast fusion,Recombinant DNA technology,C,en,Horticulture,MCQ,Medium,C,True
496,cf0eafe7-3a4c-44ff-8cfd-503eae1fed6f,Which one of the following is a mutant variety...,CO-2,Pusa nanha,Pusa Majesty,Coorg Honey Dew,B,en,Horticulture,MCQ,Medium,B,True
497,6028c963-4503-48de-b12f-163b795aa390,Which one is a sedimentary rock?,Sandstone,Basalt,Quartzite,Diorite,A,en,Soil Science,MCQ,Easy,A,True
498,71270aef-512c-431a-8d63-272de1076a01,Rhizobium strain suitable for inoculation of b...,R. trifoli,R. japonicum,R. meliloti,R. leguminosarum,A,en,Agronomy,MCQ,Medium,D,False


In [ ]:
# ============================================================
# EXISTING BENCHMARK + W&B WEAVE
# Synchronous / Google Colab
# Weave version: 0.53.11
# ============================================================

import pandas as pd
import tqdm
import wandb
import weave
import torch
import re
import uuid


# ============================================================
# WEAVE CONFIG
# ============================================================

WEAVE_PROJECT_NAME = "Agri-Benchmark-September2026"

DATASET_NAME = "AgriBenchmark"
DATASET_VERSION = "v1"

PROMPT_VERSION = "v1"
PARSER_VERSION = "v1"


# ============================================================
# INITIALIZE WEAVE
# ============================================================

weave.init(WEAVE_PROJECT_NAME)


# ============================================================
# PROMPT BUILDER
# ============================================================

def build_prompt_string(row):

    question_block = f"{row['question']}\n"

    options_block = (
        f"A: {row['option_a']}\n"
        f"B: {row['option_b']}\n"
        f"C: {row['option_c']}\n"
        f"D: {row['option_d']}\n"
    )

    final_user_content = (
        BASE_PROMPT +
        question_block +
        options_block +
        "What is the correct answer? "
        "Only provide the correct option (A/B/C/D)"
    )

    messages = [
        {
            "role": "user",
            "content": final_user_content
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


# ============================================================
# ANSWER PARSER
# ============================================================

def extract_answer(raw_text):

    # Keeping your original parsing behaviour.
    clean_ans = "".join(
        char
        for char in raw_text
        if char.isalpha()
    ).upper()

    return (
        clean_ans[0]
        if len(clean_ans) > 0
        else "INVALID"
    )


# ============================================================
# WEAVE-TRACED INFERENCE
#
# This is the only significant addition to your existing code.
# ============================================================

@weave.op()
def weave_predict(
    formatted_prompt,
    expected_answer,
    benchmark_id,
    model_name,
    dataset_name,
    dataset_version,
    prompt_version,
    parser_version,
    max_new_tokens,
    do_sample,
    temperature
):

    # --------------------------------------------------------
    # Tokenization
    # --------------------------------------------------------

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(base_model.device)

    # --------------------------------------------------------
    # Model inference
    # --------------------------------------------------------

    with torch.no_grad():

        output_ids = base_model.generate(
            **inputs,
            generation_config=generation_config
        )

    # --------------------------------------------------------
    # Remove prompt tokens
    # --------------------------------------------------------

    generated_tokens = output_ids[
        0,
        inputs.input_ids.shape[-1]:
    ]

    # --------------------------------------------------------
    # Decode
    # --------------------------------------------------------

    raw_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # --------------------------------------------------------
    # Parse answer
    # --------------------------------------------------------

    clean_ans = extract_answer(
        raw_text
    )

    # --------------------------------------------------------
    # Ground truth
    # --------------------------------------------------------

    expected = str(
        expected_answer
    ).strip().upper()

    # --------------------------------------------------------
    # Score
    # --------------------------------------------------------

    is_match = (
        clean_ans == expected
    )

    # --------------------------------------------------------
    # Everything returned here is visible in Weave
    # --------------------------------------------------------

    return {
        "prediction": clean_ans,
        "raw_output": raw_text,
        "correct_answer": expected,
        "is_correct": is_match,

        # Benchmark provenance
        "benchmark_id": benchmark_id,
        "dataset_name": dataset_name,
        "dataset_version": dataset_version,
        "model_name": model_name,
        "prompt_version": prompt_version,
        "parser_version": parser_version,

        # Generation configuration
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "temperature": temperature,
    }


# ============================================================
# BENCHMARK
# ============================================================

def run_benchmark(
    df: pd.DataFrame,
    model,
    index,
    generation_config,
    collection_name="BenchmarkResults"
):

    # ========================================================
    # MODEL INFORMATION
    # ========================================================

    model_name = model.config._name_or_path

    # ========================================================
    # UNIQUE BENCHMARK ID
    #
    # This lets you distinguish multiple evaluations of the
    # same model/configuration.
    # ========================================================

    benchmark_id = (
        f"{DATASET_NAME}-{DATASET_VERSION}"
        f"__{model_name.split('/')[-1]}"
        f"__samples-{index}"
        f"__temp-{generation_config.temperature}"
        f"__tokens-{generation_config.max_new_tokens}"
        f"__{uuid.uuid4().hex[:8]}"
    )

    # ========================================================
    # W&B RUN
    # ========================================================

    run = wandb.init(
        project=WANDB_PROJECT_NAME,
        name=(
            f"run-{model_name.split('/')[-1]}"
            f"-top-{index}"
        ),
        config={
            # Model
            "model_name": model_name,

            # Benchmark
            "benchmark_id": benchmark_id,
            "dataset_name": DATASET_NAME,
            "dataset_version": DATASET_VERSION,
            "evaluation_samples": index,

            # Prompt / parser
            "prompt_version": PROMPT_VERSION,
            "parser_version": PARSER_VERSION,

            # Generation
            "max_new_tokens":
                generation_config.max_new_tokens,

            "do_sample":
                generation_config.do_sample,

            "temperature":
                generation_config.temperature,
        }
    )

    # ========================================================
    # RESULT STORAGE
    # ========================================================

    model_answers = []
    raw_outputs = []
    matches = []

    df = df.iloc[0:index]

    print(
        f"Running modern manual inference loop "
        f"on {len(df)} rows..."
    )

    print(
        f"Benchmark ID: {benchmark_id}"
    )

    # ========================================================
    # MANUAL SYNCHRONOUS LOOP
    # ========================================================

    for _, row in tqdm.tqdm(
        df.iterrows(),
        total=len(df)
    ):

        # ----------------------------------------------------
        # Build prompt
        # ----------------------------------------------------

        formatted_prompt = build_prompt_string(
            row
        )

        # ----------------------------------------------------
        # Weave-traced inference
        #
        # No asyncio.
        # No await.
        # No event loop.
        # ----------------------------------------------------

        result = weave_predict(
            formatted_prompt=formatted_prompt,

            expected_answer=row[
                "correct_answer"
            ],

            benchmark_id=benchmark_id,

            model_name=model_name,

            dataset_name=DATASET_NAME,

            dataset_version=DATASET_VERSION,

            prompt_version=PROMPT_VERSION,

            parser_version=PARSER_VERSION,

            max_new_tokens=(
                generation_config.max_new_tokens
            ),

            do_sample=(
                generation_config.do_sample
            ),

            temperature=(
                generation_config.temperature
            ),
        )

        # ----------------------------------------------------
        # Store results locally
        # ----------------------------------------------------

        model_answers.append(
            result["prediction"]
        )

        raw_outputs.append(
            result["raw_output"]
        )

        matches.append(
            result["is_correct"]
        )

    # ========================================================
    # BUILD RESULT DATAFRAME
    # ========================================================

    df_result = df.copy()

    df_result["model_predicted"] = (
        model_answers
    )

    df_result["raw_output"] = (
        raw_outputs
    )

    df_result["is_correct"] = (
        matches
    )

    # ========================================================
    # ACCURACY
    # ========================================================

    accuracy = (
        df_result["is_correct"].sum()
        / len(df_result)
    ) * 100

    print(
        f"\n🚀 Benchmarking Complete!"
    )

    print(
        f"Model: {model_name}"
    )

    print(
        f"Dataset: "
        f"{DATASET_NAME} {DATASET_VERSION}"
    )

    print(
        f"Benchmark ID: {benchmark_id}"
    )

    print(
        f"Final Accuracy Score: "
        f"{accuracy:.2f}%"
    )

    # ========================================================
    # LOG FINAL METRIC TO W&B
    # ========================================================

    wandb.log({
        "benchmark/accuracy": accuracy,
        "benchmark/correct": (
            df_result["is_correct"].sum()
        ),
        "benchmark/incorrect": (
            (~df_result["is_correct"]).sum()
        ),
        "benchmark/samples": len(df_result),
    })

    return df_result


# ============================================================
# RUN
# ============================================================

results = run_benchmark(
    df=benchmark_dataset_raw,
    model=base_model,
    index=500,
    generation_config=generation_config
)

wandb: Initializing weave.
wandb: WARNING Failed to automatically initialize weave: Weave is already initialized for 'debasishdhal/Agri-Benchmark-September2026'; cannot initialize it for 'debasishdhal/AgriFineTuningDemo-Sep2026'.


Running modern manual inference loop on 500 rows...
Benchmark ID: AgriBenchmark-v1__Llama-3.2-3B-Instruct__samples-500__temp-1e-05__tokens-100__719753d8


100%|██████████| 500/500 [05:36<00:00,  1.49it/s]
weave: 🍩 https://wandb.ai/debasishdhal/Agri-Benchmark-September2026/r/call/01a0dc4b-5121-7d27-aeee-7de0a148ee64



🚀 Benchmarking Complete!
Model: meta-llama/Llama-3.2-3B-Instruct
Dataset: AgriBenchmark v1
Benchmark ID: AgriBenchmark-v1__Llama-3.2-3B-Instruct__samples-500__temp-1e-05__tokens-100__719753d8
Final Accuracy Score: 39.80%


In [ ]:
import os
import torch
import pandas as pd
import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel


# ============================================================
# CONFIG
# ============================================================

HF_TOKEN = hf_token

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

ADAPTER = (
    "DebasishDhal99/"
    "Llama-3.2-3B-Instruct-agriculture-qlora"
)

N_SAMPLES = 500
MAX_NEW_TOKENS = 16


# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER,
    token=HF_TOKEN,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# 4-BIT CONFIG
# ============================================================

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)


# ============================================================
# BASE MODEL
# ============================================================

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model.config.use_cache = True


# ============================================================
# LoRA ADAPTER
# ============================================================

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER,
    token=HF_TOKEN,
)

model.eval()

print("Model loaded:")
print(f"  Base:    {BASE_MODEL}")
print(f"  Adapter: {ADAPTER}")


# ============================================================
# PROMPT
# ============================================================

def build_prompt_string(row):

    question_block = (
        f"{row['question']}\n"
    )

    options_block = (
        f"A: {row['option_a']}\n"
        f"B: {row['option_b']}\n"
        f"C: {row['option_c']}\n"
        f"D: {row['option_d']}\n"
    )

    final_user_content = (
        BASE_PROMPT
        + question_block
        + options_block
        + "What is the correct answer? "
          "Only provide the correct option (A/B/C/D)"
    )

    messages = [
        {
            "role": "user",
            "content": final_user_content,
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


# ============================================================
# BENCHMARK
# ============================================================

def run_benchmark(
    df: pd.DataFrame,
    model,
    n_samples: int = 500,
):

    df = df.iloc[:n_samples].copy()

    predictions = []
    matches = []

    print(
        f"Running benchmark on {len(df)} samples..."
    )

    for _, row in tqdm.tqdm(
        df.iterrows(),
        total=len(df),
    ):

        prompt = build_prompt_string(row)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
        ).to(model.device)

        with torch.inference_mode():

            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated_tokens = output_ids[
            0,
            inputs.input_ids.shape[-1]:
        ]

        raw_text = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        ).strip()

        clean_ans = "".join(
            char
            for char in raw_text
            if char.isalpha()
        ).upper()

        clean_ans = (
            clean_ans[0]
            if clean_ans
            else "INVALID"
        )

        expected = (
            str(row["correct_answer"])
            .strip()
            .upper()
        )

        is_match = (
            clean_ans == expected
        )

        predictions.append(clean_ans)
        matches.append(is_match)

    results = df.copy()

    results["model_predicted"] = predictions
    results["is_correct"] = matches

    accuracy = (
        results["is_correct"].mean()
        * 100
    )

    print()
    print("=" * 50)
    print(
        f"Accuracy: {accuracy:.2f}%"
    )
    print(
        f"Correct: "
        f"{results['is_correct'].sum()} "
        f"/ {len(results)}"
    )
    print("=" * 50)

    return results


# ============================================================
# RUN
# ============================================================

results = run_benchmark(
    df=benchmark_dataset_raw,
    model=model,
    n_samples=N_SAMPLES,
)

print(results)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model loaded:
  Base:    meta-llama/Llama-3.2-3B-Instruct
  Adapter: DebasishDhal99/Llama-3.2-3B-Instruct-agriculture-qlora
Running benchmark on 500 samples...


100%|██████████| 500/500 [08:27<00:00,  1.01s/it]


Accuracy: 33.80%
Correct: 169 / 500
                                       id  \
0    80a5aea2-be15-4b6d-b98f-4b0eca30f50e   
1    6e6014d1-5f74-4192-ab52-b037ca469844   
2    dfdc695b-c94f-4a0d-a002-9b71e4cdbcd1   
3    0a70305f-d7a8-4723-998a-4ca2b3293967   
4    eea84ced-5c3a-4f26-87e1-4574743beb48   
..                                    ...   
495  49b14875-1296-4b37-adb1-1052cf87319a   
496  cf0eafe7-3a4c-44ff-8cfd-503eae1fed6f   
497  6028c963-4503-48de-b12f-163b795aa390   
498  71270aef-512c-431a-8d63-272de1076a01   
499  ee7028bd-a53c-4492-b45b-96a6b69135dc   

                                              question  \
0    It is state or condition of atmosphere at give...   
1    It refers to the water vapour content of the a...   
2                        Wind velocity is measured by?   
3    It is an aggregate of minute drops of water su...   
4    It is an automated method of surface irrigation.?   
..                                                 ...   
495         Soma

In [ ]:
import os
import torch
import pandas as pd
import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel


# ============================================================
# CONFIG
# ============================================================

HF_TOKEN = hf_token

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

ADAPTER = (
    "DebasishDhal99/"
    "Llama-3.2-3B-Instruct-agriculture-lora"
)

N_SAMPLES = 500
MAX_NEW_TOKENS = 16


# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER,
    token=HF_TOKEN,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token



# ============================================================
# BASE MODEL
# ============================================================
DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    # quantization_config=bnb_config,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)

base_model.config.use_cache = True


# ============================================================
# LoRA ADAPTER
# ============================================================

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER,
    token=HF_TOKEN,
)

model.eval()

print("Model loaded:")
print(f"  Base:    {BASE_MODEL}")
print(f"  Adapter: {ADAPTER}")


# ============================================================
# PROMPT
# ============================================================

def build_prompt_string(row):

    question_block = (
        f"{row['question']}\n"
    )

    options_block = (
        f"A: {row['option_a']}\n"
        f"B: {row['option_b']}\n"
        f"C: {row['option_c']}\n"
        f"D: {row['option_d']}\n"
    )

    final_user_content = (
        BASE_PROMPT
        + question_block
        + options_block
        + "What is the correct answer? "
          "Only provide the correct option (A/B/C/D)"
    )

    messages = [
        {
            "role": "user",
            "content": final_user_content,
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


# ============================================================
# BENCHMARK
# ============================================================

def run_benchmark(
    df: pd.DataFrame,
    model,
    n_samples: int = 500,
):

    df = df.iloc[:n_samples].copy()

    predictions = []
    matches = []

    print(
        f"Running benchmark on {len(df)} samples..."
    )

    for _, row in tqdm.tqdm(
        df.iterrows(),
        total=len(df),
    ):

        prompt = build_prompt_string(row)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
        ).to(model.device)

        with torch.inference_mode():

            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated_tokens = output_ids[
            0,
            inputs.input_ids.shape[-1]:
        ]

        raw_text = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        ).strip()

        clean_ans = "".join(
            char
            for char in raw_text
            if char.isalpha()
        ).upper()

        clean_ans = (
            clean_ans[0]
            if clean_ans
            else "INVALID"
        )

        expected = (
            str(row["correct_answer"])
            .strip()
            .upper()
        )

        is_match = (
            clean_ans == expected
        )

        predictions.append(clean_ans)
        matches.append(is_match)

    results = df.copy()

    results["model_predicted"] = predictions
    results["is_correct"] = matches

    accuracy = (
        results["is_correct"].mean()
        * 100
    )

    print()
    print("=" * 50)
    print(
        f"Accuracy: {accuracy:.2f}%"
    )
    print(
        f"Correct: "
        f"{results['is_correct'].sum()} "
        f"/ {len(results)}"
    )
    print("=" * 50)

    return results


# ============================================================
# RUN
# ============================================================

results = run_benchmark(
    df=benchmark_dataset_raw,
    model=model,
    n_samples=N_SAMPLES,
)

print(results)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 97.3MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Model loaded:
  Base:    meta-llama/Llama-3.2-3B-Instruct
  Adapter: DebasishDhal99/Llama-3.2-3B-Instruct-agriculture-lora
Running benchmark on 500 samples...


100%|██████████| 500/500 [06:59<00:00,  1.19it/s]


Accuracy: 35.00%
Correct: 175 / 500
                                       id  \
0    80a5aea2-be15-4b6d-b98f-4b0eca30f50e   
1    6e6014d1-5f74-4192-ab52-b037ca469844   
2    dfdc695b-c94f-4a0d-a002-9b71e4cdbcd1   
3    0a70305f-d7a8-4723-998a-4ca2b3293967   
4    eea84ced-5c3a-4f26-87e1-4574743beb48   
..                                    ...   
495  49b14875-1296-4b37-adb1-1052cf87319a   
496  cf0eafe7-3a4c-44ff-8cfd-503eae1fed6f   
497  6028c963-4503-48de-b12f-163b795aa390   
498  71270aef-512c-431a-8d63-272de1076a01   
499  ee7028bd-a53c-4492-b45b-96a6b69135dc   

                                              question  \
0    It is state or condition of atmosphere at give...   
1    It refers to the water vapour content of the a...   
2                        Wind velocity is measured by?   
3    It is an aggregate of minute drops of water su...   
4    It is an automated method of surface irrigation.?   
..                                                 ...   
495         Soma

In [ ]:
import os
import re
import json
import torch
import pandas as pd
import wandb
from peft import PeftModel
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

# ============================================================
# CONFIG
# ============================================================

# HF_TOKEN = os.environ["HF_TOKEN"]
# WANDB_PROJECT = "agriculture-finetuning"

bch_dataset_id = "DebasishDhal99/agriculture-synthetic-finetuning-dataset"
ft_dataset_id = "DebasishDhal99/agriculture-synthetic-finetuning-dataset"
base_model_id = "meta-llama/Llama-3.2-3B-Instruct"
model_name = base_model_id.split("/")[-1]

# base_model_id = "meta-llama/Llama-3.2-3B-Instruct"
# model_name = base_model_id.split("/")[-1]

DATASET_CONFIG = "v0"
BENCHMARK_SPLIT = "benchmark_hard"

MAX_NEW_TOKENS = 8

# ============================================================
# WANDB
# ============================================================

wandb.init(
    project=WANDB_PROJECT_NAME,
    name=f"lora-hardFineTunedLlama3BInstruct-benchmark-hard_synthetic-data-{model_name}",
    config={
        "model": base_model_id,
        "dataset": bch_dataset_id,
        "dataset_config": DATASET_CONFIG,
        "benchmark_split": BENCHMARK_SPLIT,
        "max_new_tokens": MAX_NEW_TOKENS,
    },
)

ADAPTER_ID = "DebasishDhal99/Llama-3.2-3B-Instruct-synthetic-agriculture-lora"

# ============================================================
# LOAD DATASET
# ============================================================

dataset = load_dataset(
    bch_dataset_id,
    name=DATASET_CONFIG,
    token=hf_token,
)

benchmark = dataset[BENCHMARK_SPLIT]

print(dataset)
print(f"Benchmark examples: {len(benchmark)}")

# ============================================================
# INSPECT COLUMNS
# ============================================================

print("\nColumns:")
print(benchmark.column_names)

print("\nFirst example:")
print(benchmark[0])

# ============================================================
# LOAD MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    token=hf_token,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    token=hf_token,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    model,
    ADAPTER_ID,
    token=hf_token,
)

model.eval()

# Llama doesn't normally have a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ============================================================
# HELPERS
# ============================================================

def get_question(row):
    """
    Supports the CSV structure generated previously:

        question
        answer
        answer_text

    and also ChatML-style messages if present.
    """

    # Case 1: explicit question column
    if "question" in row:
        return row["question"]

    # Case 2: messages column
    if "messages" in row:
        messages = row["messages"]

        if isinstance(messages, str):
            messages = json.loads(messages)

        for message in messages:
            if message["role"] == "user":
                return message["content"]

    raise ValueError(
        f"Cannot find question column. Available columns: {row.keys()}"
    )


def get_expected_answer(row):
    """
    Expected answer should be A/B/C/D.
    """

    answer = row.get("answer", None)

    if answer is None:
        raise ValueError(
            f"Cannot find answer column. Available columns: {row.keys()}"
        )

    answer = str(answer).strip().upper()

    # Extract A/B/C/D if answer contains additional text
    match = re.search(r"\b([ABCD])\b", answer)

    if match:
        return match.group(1)

    raise ValueError(f"Could not parse expected answer: {answer}")


def extract_prediction(text):
    """
    Extract A/B/C/D from model output.

    We strongly prefer the first standalone A/B/C/D.
    """

    text = text.strip().upper()

    # First try a standalone letter
    match = re.search(r"\b([ABCD])\b", text)

    if match:
        return match.group(1)

    # Handle outputs such as:
    # "A."
    # "(A)"
    # "Answer: A"
    match = re.search(r"(?:^|[\s:(])([ABCD])(?:[.)\s]|$)", text)

    if match:
        return match.group(1)

    return None


# ============================================================
# EVALUATION
# ============================================================

results = []

correct = 0
invalid = 0

print("\nStarting benchmark...\n")

for idx, row in enumerate(benchmark):

    question = get_question(row)
    expected = get_expected_answer(row)

    messages = [
        {
            "role": "system",
            "content": (
                "You are an agricultural question answering assistant. "
                "Answer the multiple-choice question by returning only "
                "the letter A, B, C, or D."
            ),
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    prediction = extract_prediction(generated_text)

    is_correct = prediction == expected

    if prediction is None:
        invalid += 1
    elif is_correct:
        correct += 1

    results.append({
        "index": idx,
        "question": question,
        "expected": expected,
        "prediction": prediction,
        "raw_output": generated_text,
        "correct": is_correct,
    })

    print(
        f"[{idx + 1:03d}/{len(benchmark)}] "
        f"Expected={expected} "
        f"Predicted={prediction} "
        f"Correct={is_correct}"
    )


# ============================================================
# METRICS
# ============================================================

total = len(results)

accuracy = correct / total
invalid_rate = invalid / total

print("\n" + "=" * 60)
print("BASE MODEL BENCHMARK")
print("=" * 60)

print(f"Model:             {base_model_id}")
print(f"Benchmark size:    {total}")
print(f"Correct:           {correct}")
print(f"Incorrect:         {total - correct - invalid}")
print(f"Invalid:           {invalid}")
print(f"Accuracy:          {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Invalid rate:      {invalid_rate:.4f} ({invalid_rate * 100:.2f}%)")

# ============================================================
# WANDB LOGGING
# ============================================================

wandb.log({
    "benchmark_size": total,
    "correct": correct,
    "incorrect": total - correct - invalid,
    "invalid": invalid,
    "accuracy": accuracy,
    "invalid_rate": invalid_rate,
})

# ============================================================
# SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_path = "base_benchmark_results.csv"

results_df.to_csv(
    results_path,
    index=False,
)

print(f"\nResults saved to: {results_path}")

# Log prediction table to W&B
wandb_table = wandb.Table(
    dataframe=results_df
)

wandb.log({
    "benchmark_predictions": wandb_table
})

wandb.finish()

DatasetDict({
    train: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 100
    })
    benchmark: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 80
    })
    benchmark_hard: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 300
    })
})
Benchmark examples: 300

Columns:
['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text']

First example:
{'id': 'benchmark_0001', 'crop': 'maize', 'category': 'nutrient', 'condition': 'Nitrogen deficiency', 'question': 'Crop: Maize\nField observations: older leaves show yellow-green V-shaped discoloration beginning near the tip and extending toward the midr

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 97.3MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            


Starting benchmark...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[001/300] Expected=B Predicted=B Correct=True
[002/300] Expected=D Predicted=D Correct=True
[003/300] Expected=A Predicted=A Correct=True
[004/300] Expected=C Predicted=C Correct=True
[005/300] Expected=A Predicted=A Correct=True
[006/300] Expected=C Predicted=C Correct=True
[007/300] Expected=A Predicted=A Correct=True
[008/300] Expected=C Predicted=A Correct=False
[009/300] Expected=B Predicted=B Correct=True
[010/300] Expected=B Predicted=B Correct=True
[011/300] Expected=D Predicted=D Correct=True
[012/300] Expected=D Predicted=D Correct=True
[013/300] Expected=A Predicted=A Correct=True
[014/300] Expected=B Predicted=B Correct=True
[015/300] Expected=A Predicted=A Correct=True
[016/300] Expected=C Predicted=C Correct=True
[017/300] Expected=A Predicted=A Correct=True
[018/300] Expected=B Predicted=B Correct=True
[019/300] Expected=C Predicted=C Correct=True
[020/300] Expected=A Predicted=A Correct=True
[021/300] Expected=D Predicted=D Correct=True
[022/300] Expected=A Predicted=A 

accuracy,▁
benchmark_size,▁
correct,▁
incorrect,▁
invalid,▁
invalid_rate,▁
accuracy,0.90333
benchmark_size,300
correct,271
incorrect,29
invalid,0


# Fine-tuning

In [ ]:
!pip -q install -U transformers datasets trl peft accelerate bitsandbytes wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.2/927.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is 

In [ ]:
# ============================================================
# AGRICULTURE SFT — KisanVaani → Qwen2.5-3B-Instruct
# QLoRA / Google Colab
# ============================================================



import os
import re
import torch
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_NAME = base_model_id

DATASET_NAME = "KisanVaani/agriculture-qa-english-only"

OUTPUT_DIR = "./meta-llama-3p5b-instruct-agriculture-lora"

SEED = 42

# Training
EPOCHS = 2
LEARNING_RATE = 2e-4

BATCH_SIZE = 2
GRAD_ACCUMULATION = 8

MAX_LENGTH = 1024

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


# ============================================================
# 1. LOAD DATASET
# ============================================================

dataset = load_dataset(
    DATASET_NAME,
    split="train"
)

print(dataset)
print(dataset.column_names)


# ============================================================
# 2. BASIC DATA AUDIT
# ============================================================

df = dataset.to_pandas()

print("\nShape:", df.shape)
print("\nMissing:")
print(df.isna().sum())

print(
    "\nDuplicate Q/A:",
    df.duplicated(
        subset=["question", "answers"]
    ).sum()
)

print(
    "\nDuplicate questions:",
    df["question"].duplicated().sum()
)


# ============================================================
# 3. CLEAN DATA
# ============================================================

df["question"] = (
    df["question"]
    .astype(str)
    .str.strip()
)

df["answers"] = (
    df["answers"]
    .astype(str)
    .str.strip()
)

# Remove empty / invalid rows
df = df[
    (df["question"].str.len() > 5) &
    (df["answers"].str.len() > 2)
]

# Remove exact duplicates
df = df.drop_duplicates(
    subset=["question", "answers"]
)

# Remove duplicate questions
df = df.drop_duplicates(
    subset=["question"]
)

df = df.reset_index(drop=True)

print(
    "\nClean dataset:",
    df.shape
)


# ============================================================
# 4. CONVERT TO CHAT FORMAT
# ============================================================

def convert_to_messages(example):

    return {
        "messages": [
            {
                "role": "user",
                "content": example["question"],
            },
            {
                "role": "assistant",
                "content": example["answers"],
            },
        ]
    }


clean_dataset = load_dataset(
    "KisanVaani/agriculture-qa-english-only",
    split="train"
)

# Rebuild from cleaned dataframe
from datasets import Dataset

clean_dataset = Dataset.from_pandas(
    df,
    preserve_index=False
)

clean_dataset = clean_dataset.map(
    convert_to_messages,
    remove_columns=clean_dataset.column_names
)


# ============================================================
# 5. TRAIN / VALIDATION SPLIT
# ============================================================

split = clean_dataset.train_test_split(
    test_size=0.10,
    seed=SEED
)

train_dataset = split["train"]
eval_dataset = split["test"]

print(
    f"\nTrain: {len(train_dataset)}"
)

print(
    f"Eval : {len(eval_dataset)}"
)


# ============================================================
# 6. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=hf_token
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# 7. 4-BIT QUANTIZATION
# ============================================================

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)


# ============================================================
# 8. LOAD MODEL
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=hf_token
)

model.config.use_cache = False

import wandb

wandb.init(
    project=WANDB_PROJECT_NAME,
    name=f"{base_model_id}-agriculture-qlora",
    config={
        "base_model": MODEL_NAME,
        "dataset": DATASET_NAME,
        "dataset_version": "v1",
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUMULATION,
        "max_length": MAX_LENGTH,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
    }
)
# ============================================================
# 9. LoRA
# ============================================================

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",
    task_type="CAUSAL_LM",
)


# ============================================================
# 10. TRAINING CONFIG
# ============================================================

training_args = SFTConfig(

    output_dir=OUTPUT_DIR,

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUMULATION,

    learning_rate=LEARNING_RATE,

    max_length=MAX_LENGTH,

    gradient_checkpointing=False,

    eval_strategy="steps",
    eval_steps=250,

    save_strategy="steps",
    save_steps=250,

    logging_steps=10,

    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    report_to="wandb",

    run_name=f"{base_model_id}-agriculture-qlora",

    seed=SEED,
)


# ============================================================
# 11. TRAINER
# ============================================================

trainer = SFTTrainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    processing_class=tokenizer,

    peft_config=peft_config,
)


# ============================================================
# 12. TRAIN
# ============================================================

trainer.train()


# ============================================================
# 13. SAVE LoRA ADAPTER
# ============================================================

trainer.save_model(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)


# ============================================================
# 14. FINAL EVALUATION LOSS
# ============================================================

eval_results = trainer.evaluate()

print("\nFinal evaluation:")
print(eval_results)


# ============================================================
# 15. OPTIONAL: TEST GENERATION
# ============================================================

test_question = (
    "What are the benefits of crop rotation?"
)

messages = [
    {
        "role": "user",
        "content": test_question,
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

generated = output[
    0,
    inputs.input_ids.shape[-1]:
]

answer = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print("\nQuestion:")
print(test_question)

print("\nModel:")
print(answer)

Dataset({
    features: ['question', 'answers'],
    num_rows: 22615
})
['question', 'answers']

Shape: (22615, 2)

Missing:
question    0
answers     0
dtype: int64

Duplicate Q/A: 20284

Duplicate questions: 20390

Clean dataset: (2222, 2)


Map:   0%|          | 0/2222 [00:00<?, ? examples/s]


Train: 1999
Eval : 223


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

...

Truncating eval dataset:   0%|          | 0/223 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/223 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
!pip install -U transformers trl peft accelerate torchao

In [ ]:
import transformers
import trl
import peft, torchao
import accelerate

print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("torchao:", torchao.__version__)

transformers: 5.16.1
trl: 1.14.1
peft: 0.20.0
accelerate: 1.14.0
torchao: 0.10.0


In [ ]:
# ============================================================
# QLoRA SFT — Agriculture
# Qwen2.5-3B-Instruct + KisanVaani
# W&B + Hugging Face Hub
# ============================================================

import os
import logging
from pathlib import Path

import torch
import wandb

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig


# ============================================================
# CONFIG
# ============================================================

HF_TOKEN = hf_token
WANDB_API_KEY = os.environ["WANDB_API_KEY"]

MODEL_NAME = base_model_id

TRAIN_DATASET = "KisanVaani/agriculture-qa-english-only"

# Set this to your actual benchmark later
BENCHMARK_DATASET = None

HF_REPO_ID = f"DebasishDhal99/{model_name}-agriculture-qlora"

WANDB_PROJECT = WANDB_PROJECT_NAME
RUN_NAME = f"{model_name}-lora"

OUTPUT_DIR = f"./{model_name}-agriculture-lora"

SEED = 42

EPOCHS = 2
LEARNING_RATE = 2e-4

BATCH_SIZE = 2
GRAD_ACCUMULATION = 8

MAX_LENGTH = 1024

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)


# ============================================================
# W&B
# ============================================================

wandb.login(key=WANDB_API_KEY)

wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    config={
        "base_model": MODEL_NAME,
        "train_dataset": TRAIN_DATASET,
        "benchmark_dataset": BENCHMARK_DATASET,

        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUMULATION,
        "max_length": MAX_LENGTH,

        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,

        "seed": SEED,
    },
)


# ============================================================
# DATA
# ============================================================

dataset = load_dataset(
    TRAIN_DATASET,
    split="train",
    token=HF_TOKEN,
)

logger.info("Raw dataset: %d examples", len(dataset))

df = dataset.to_pandas()

# Clean
df["question"] = (
    df["question"]
    .astype(str)
    .str.strip()
)

df["answers"] = (
    df["answers"]
    .astype(str)
    .str.strip()
)

df = df[
    (df["question"].str.len() > 5)
    &
    (df["answers"].str.len() > 2)
]

df = df.drop_duplicates(
    subset=["question"]
).reset_index(drop=True)

dataset = Dataset.from_pandas(
    df,
    preserve_index=False,
)


# ============================================================
# CHAT FORMAT
# ============================================================

def format_example(example):

    return {
        "messages": [
            {
                "role": "user",
                "content": example["question"],
            },
            {
                "role": "assistant",
                "content": example["answers"],
            },
        ]
    }


dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names,
)

split = dataset.train_test_split(
    test_size=0.10,
    seed=SEED,
)

train_dataset = split["train"]
eval_dataset = split["test"]

logger.info(
    "Train: %d | Eval: %d",
    len(train_dataset),
    len(eval_dataset),
)

wandb.config.update({
    "train_examples": len(train_dataset),
    "eval_examples": len(eval_dataset),
})


# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# 4-BIT QUANTIZATION
# ============================================================

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)


# ============================================================
# MODEL
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False


# ============================================================
# LoRA
# ============================================================

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",
    task_type="CAUSAL_LM",
)


# ============================================================
# TRAINING
# ============================================================

training_args = SFTConfig(

    output_dir=OUTPUT_DIR,

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUMULATION,

    learning_rate=LEARNING_RATE,

    max_length=MAX_LENGTH,

    weight_decay=0.01,
    # warmup_ratio=0.03,

    eval_strategy="steps",
    eval_steps=250,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_strategy="steps",
    logging_steps=10,

    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    gradient_checkpointing=False,

    report_to="wandb",
    run_name=RUN_NAME,

    seed=SEED,
    data_seed=SEED,
)


# ============================================================
# TRAINER
# ============================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    processing_class=tokenizer,

    peft_config=peft_config,
)


# ============================================================
# TRAIN
# ============================================================

logger.info("Starting training...")

train_result = trainer.train()

logger.info(
    "Training complete: %s",
    train_result.metrics,
)


# ============================================================
# BEST MODEL
# ============================================================

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

eval_results = trainer.evaluate()

logger.info(
    "Final evaluation: %s",
    eval_results,
)

wandb.log({
    f"final/{k}": v
    for k, v in eval_results.items()
    if isinstance(v, (int, float))
})


# ============================================================
# W&B MODEL ARTIFACT
# ============================================================

artifact = wandb.Artifact(
    name=f"{RUN_NAME}-adapter",
    type="model",
    metadata={
        "base_model": MODEL_NAME,
        "dataset": TRAIN_DATASET,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
    },
)

artifact.add_dir(OUTPUT_DIR)

wandb.log_artifact(artifact)


# ============================================================
# HUGGING FACE
# ============================================================

logger.info(
    "Pushing adapter to %s",
    HF_REPO_ID,
)

trainer.model.push_to_hub(
    HF_REPO_ID,
    token=HF_TOKEN,
)

tokenizer.push_to_hub(
    HF_REPO_ID,
    token=HF_TOKEN,
)

logger.info("Done.")

wandb.finish()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Map:   0%|          | 0/2222 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/1999 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1999 [00:00<?, ? examples/s]

...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpu30t9f5c/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

eval/entropy,▁▁
eval/loss,▁▁
eval/mean_token_accuracy,▁▁
eval/num_tokens,▁▁
eval/runtime,▁█
eval/samples_per_second,█▁
eval/steps_per_second,█▁
final/eval_entropy,▁
final/eval_loss,▁
final/eval_mean_token_accuracy,▁
+9,...


In [ ]:
# ============================================================
# LoRA SFT — Agriculture
# Base model + KisanVaani
# NO QUANTIZATION
# Adapter-only output
# W&B + Hugging Face Hub
# ============================================================

import os
import logging
import time
import platform

import torch
import wandb

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig


# ============================================================
# CONFIG
# ============================================================

HF_TOKEN = hf_token
WANDB_API_KEY = os.environ["WANDB_API_KEY"]

MODEL_NAME = base_model_id

TRAIN_DATASET = "KisanVaani/agriculture-qa-english-only"

HF_REPO_ID = f"DebasishDhal99/{model_name}-agriculture-lora"
WANDB_PROJECT = WANDB_PROJECT_NAME
RUN_NAME = f"{model_name}-agriculture-lora"

OUTPUT_DIR = f"./{model_name}-agriculture-lora"

SEED = 42

EPOCHS = 2
LEARNING_RATE = 2e-4

BATCH_SIZE = 2
GRAD_ACCUMULATION = 8

MAX_LENGTH = 1024

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)


# ============================================================
# DTYPE
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required for this training setup.")

USE_BF16 = torch.cuda.is_bf16_supported()

DTYPE = (
    torch.bfloat16
    if USE_BF16
    else torch.float16
)

logger.info("Using dtype: %s", DTYPE)


# ============================================================
# W&B
# ============================================================

wandb.login(key=WANDB_API_KEY)

wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    config={
        "base_model": MODEL_NAME,
        "training_method": "LoRA",
        "quantization": "none",
        "dtype": str(DTYPE),

        "train_dataset": TRAIN_DATASET,

        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUMULATION,
        "effective_batch_size": BATCH_SIZE * GRAD_ACCUMULATION,
        "max_length": MAX_LENGTH,

        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,

        "seed": SEED,

        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
    },
)


# ============================================================
# DATASET
# ============================================================

dataset = load_dataset(
    TRAIN_DATASET,
    split="train",
    token=HF_TOKEN,
)

raw_examples = len(dataset)

logger.info(
    "Raw examples: %d",
    raw_examples,
)

df = dataset.to_pandas()

df["question"] = (
    df["question"]
    .astype(str)
    .str.strip()
)

df["answers"] = (
    df["answers"]
    .astype(str)
    .str.strip()
)

before_cleaning = len(df)

df = df[
    (df["question"].str.len() > 5)
    &
    (df["answers"].str.len() > 2)
]

after_filter = len(df)

df = df.drop_duplicates(
    subset=["question"]
).reset_index(drop=True)

clean_examples = len(df)

logger.info(
    "Clean examples: %d",
    clean_examples,
)

wandb.config.update({
    "raw_examples": raw_examples,
    "clean_examples": clean_examples,
    "removed_examples": before_cleaning - clean_examples,
})


dataset = Dataset.from_pandas(
    df,
    preserve_index=False,
)


# ============================================================
# CHAT FORMAT
# ============================================================

def format_example(example):

    return {
        "messages": [
            {
                "role": "user",
                "content": example["question"],
            },
            {
                "role": "assistant",
                "content": example["answers"],
            },
        ]
    }


dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names,
)


# ============================================================
# TRAIN / VALIDATION SPLIT
# ============================================================

split = dataset.train_test_split(
    test_size=0.10,
    seed=SEED,
)

train_dataset = split["train"]
eval_dataset = split["test"]

logger.info(
    "Train: %d | Eval: %d",
    len(train_dataset),
    len(eval_dataset),
)

wandb.config.update({
    "train_examples": len(train_dataset),
    "eval_examples": len(eval_dataset),
})


# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# MODEL
# ============================================================
# IMPORTANT:
# - No BitsAndBytesConfig
# - No 4-bit
# - No 8-bit
# - No quantization
#
# Base model is loaded directly in BF16/FP16.
# ============================================================

logger.info(
    "Loading base model WITHOUT quantization..."
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False


# ============================================================
# LoRA CONFIG
# ============================================================

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",
    task_type="CAUSAL_LM",
)


# ============================================================
# TRAINING CONFIG
# ============================================================

training_args = SFTConfig(

    output_dir=OUTPUT_DIR,

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUMULATION,

    learning_rate=LEARNING_RATE,

    max_length=MAX_LENGTH,

    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=250,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_strategy="steps",
    logging_steps=10,

    bf16=USE_BF16,
    fp16=not USE_BF16,

    gradient_checkpointing=False,

    report_to="wandb",
    run_name=RUN_NAME,

    seed=SEED,
    data_seed=SEED,
)


# ============================================================
# TRAINER
# ============================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    processing_class=tokenizer,

    peft_config=peft_config,
)


# ============================================================
# VERIFY TRAINABLE PARAMETERS
# ============================================================

trainable_params = sum(
    p.numel()
    for p in trainer.model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in trainer.model.parameters()
)

trainable_percentage = (
    100 * trainable_params / total_params
)

logger.info(
    "Total parameters: %.2f B",
    total_params / 1e9,
)

logger.info(
    "Trainable parameters: %.2f M",
    trainable_params / 1e6,
)

logger.info(
    "Trainable percentage: %.4f%%",
    trainable_percentage,
)

wandb.config.update({
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "trainable_percentage": trainable_percentage,
})


# ============================================================
# TRAIN
# ============================================================

logger.info("Starting LoRA training...")

start_time = time.time()

train_result = trainer.train()

training_time = time.time() - start_time

logger.info(
    "Training finished in %.2f minutes",
    training_time / 60,
)

wandb.log({
    "training/time_minutes": training_time / 60,
})


# ============================================================
# SAVE ADAPTER
# ============================================================
# IMPORTANT:
# save_model() with PEFT saves the adapter.
# It does NOT save a merged full model.
# ============================================================

logger.info(
    "Saving LoRA adapter to %s",
    OUTPUT_DIR,
)

trainer.save_model(OUTPUT_DIR)

tokenizer.save_pretrained(
    OUTPUT_DIR
)


# ============================================================
# FINAL VALIDATION
# ============================================================

eval_results = trainer.evaluate()

logger.info(
    "Final validation: %s",
    eval_results,
)

wandb.log({
    f"final_eval/{key}": value
    for key, value in eval_results.items()
    if isinstance(value, (int, float))
})


# ============================================================
# PERPLEXITY
# ============================================================

if "eval_loss" in eval_results:

    perplexity = torch.exp(
        torch.tensor(eval_results["eval_loss"])
    ).item()

    logger.info(
        "Validation perplexity: %.4f",
        perplexity,
    )

    wandb.log({
        "final_eval/perplexity": perplexity
    })


# ============================================================
# W&B ADAPTER ARTIFACT
# ============================================================

artifact = wandb.Artifact(
    name=f"{RUN_NAME}-adapter",
    type="model",

    metadata={
        "base_model": MODEL_NAME,
        "dataset": TRAIN_DATASET,

        "training_method": "LoRA",
        "quantization": "none",
        "dtype": str(DTYPE),

        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,

        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUMULATION,

        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,

        "max_length": MAX_LENGTH,
        "seed": SEED,
    },
)

artifact.add_dir(
    OUTPUT_DIR
)

wandb.log_artifact(
    artifact
)


# ============================================================
# HUGGING FACE — ADAPTER ONLY
# ============================================================

logger.info(
    "Uploading LoRA adapter to %s",
    HF_REPO_ID,
)

trainer.model.push_to_hub(
    HF_REPO_ID,
    token=HF_TOKEN,
)

tokenizer.push_to_hub(
    HF_REPO_ID,
    token=HF_TOKEN,
)

logger.info(
    "LoRA adapter uploaded successfully."
)


# ============================================================
# FINISH
# ============================================================

wandb.finish()

logger.info("Experiment complete.")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Map:   0%|          | 0/2222 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/1999 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1999 [00:00<?, ? examples/s]

...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp86p12347/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

eval/entropy,▁▁
eval/loss,▁▁
eval/mean_token_accuracy,▁▁
eval/num_tokens,▁▁
eval/runtime,▁█
eval/samples_per_second,█▁
eval/steps_per_second,█▁
final_eval/eval_entropy,▁
final_eval/eval_loss,▁
final_eval/eval_mean_token_accuracy,▁
+11,...


# Round 2 - Fine-tuning with synthetic Agricultural dataset

In [ ]:
bch_dataset_id = "DebasishDhal99/agriculture-synthetic-finetuning-dataset" #https://huggingface.co/datasets/DebasishDhal99/agriculture-synthetic-finetuning-dataset
ft_dataset_id = "DebasishDhal99/agriculture-synthetic-finetuning-dataset" #https://huggingface.co/datasets/DebasishDhal99/agriculture-synthetic-finetuning-dataset
base_model_id = "meta-llama/Llama-3.2-3B-Instruct"
model_name = base_model_id.split("/")[-1]

In [ ]:
import torch, gc
gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

### Baseline benchmark

In [ ]:
import os
import re
import json
import torch
import pandas as pd
import wandb

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

# ============================================================
# CONFIG
# ============================================================

# HF_TOKEN = os.environ["HF_TOKEN"]
# WANDB_PROJECT = "agriculture-finetuning"

# bch_dataset_id = "DebasishDhal99/agriculture-synthetic-finetuning-dataset"
# ft_dataset_id = "DebasishDhal99/agriculture-synthetic-finetuning-dataset"

# base_model_id = "meta-llama/Llama-3.2-3B-Instruct"
# model_name = base_model_id.split("/")[-1]

DATASET_CONFIG = "v0"
BENCHMARK_SPLIT = "benchmark_hard"

MAX_NEW_TOKENS = 8

# ============================================================
# WANDB
# ============================================================

wandb.init(
    project=WANDB_PROJECT_NAME,
    name=f"base-benchmark-hard_synthetic-data-{model_name}",
    config={
        "model": base_model_id,
        "dataset": bch_dataset_id,
        "dataset_config": DATASET_CONFIG,
        "benchmark_split": BENCHMARK_SPLIT,
        "max_new_tokens": MAX_NEW_TOKENS,
    },
)

# ============================================================
# LOAD DATASET
# ============================================================

dataset = load_dataset(
    bch_dataset_id,
    name=DATASET_CONFIG,
    token=hf_token,
)

benchmark = dataset[BENCHMARK_SPLIT]

print(dataset)
print(f"Benchmark examples: {len(benchmark)}")

# ============================================================
# INSPECT COLUMNS
# ============================================================

print("\nColumns:")
print(benchmark.column_names)

print("\nFirst example:")
print(benchmark[0])

# ============================================================
# LOAD MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    token=hf_token,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    token=hf_token,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.eval()

# Llama doesn't normally have a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ============================================================
# HELPERS
# ============================================================

def get_question(row):
    """
    Supports the CSV structure generated previously:

        question
        answer
        answer_text

    and also ChatML-style messages if present.
    """

    # Case 1: explicit question column
    if "question" in row:
        return row["question"]

    # Case 2: messages column
    if "messages" in row:
        messages = row["messages"]

        if isinstance(messages, str):
            messages = json.loads(messages)

        for message in messages:
            if message["role"] == "user":
                return message["content"]

    raise ValueError(
        f"Cannot find question column. Available columns: {row.keys()}"
    )


def get_expected_answer(row):
    """
    Expected answer should be A/B/C/D.
    """

    answer = row.get("answer", None)

    if answer is None:
        raise ValueError(
            f"Cannot find answer column. Available columns: {row.keys()}"
        )

    answer = str(answer).strip().upper()

    # Extract A/B/C/D if answer contains additional text
    match = re.search(r"\b([ABCD])\b", answer)

    if match:
        return match.group(1)

    raise ValueError(f"Could not parse expected answer: {answer}")


def extract_prediction(text):
    """
    Extract A/B/C/D from model output.

    We strongly prefer the first standalone A/B/C/D.
    """

    text = text.strip().upper()

    # First try a standalone letter
    match = re.search(r"\b([ABCD])\b", text)

    if match:
        return match.group(1)

    # Handle outputs such as:
    # "A."
    # "(A)"
    # "Answer: A"
    match = re.search(r"(?:^|[\s:(])([ABCD])(?:[.)\s]|$)", text)

    if match:
        return match.group(1)

    return None


# ============================================================
# EVALUATION
# ============================================================

results = []

correct = 0
invalid = 0

print("\nStarting benchmark...\n")

for idx, row in enumerate(benchmark):

    question = get_question(row)
    expected = get_expected_answer(row)

    messages = [
        {
            "role": "system",
            "content": (
                "You are an agricultural question answering assistant. "
                "Answer the multiple-choice question by returning only "
                "the letter A, B, C, or D."
            ),
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    prediction = extract_prediction(generated_text)

    is_correct = prediction == expected

    if prediction is None:
        invalid += 1
    elif is_correct:
        correct += 1

    results.append({
        "index": idx,
        "question": question,
        "expected": expected,
        "prediction": prediction,
        "raw_output": generated_text,
        "correct": is_correct,
    })

    print(
        f"[{idx + 1:03d}/{len(benchmark)}] "
        f"Expected={expected} "
        f"Predicted={prediction} "
        f"Correct={is_correct}"
    )


# ============================================================
# METRICS
# ============================================================

total = len(results)

accuracy = correct / total
invalid_rate = invalid / total

print("\n" + "=" * 60)
print("BASE MODEL BENCHMARK")
print("=" * 60)

print(f"Model:             {base_model_id}")
print(f"Benchmark size:    {total}")
print(f"Correct:           {correct}")
print(f"Incorrect:         {total - correct - invalid}")
print(f"Invalid:           {invalid}")
print(f"Accuracy:          {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Invalid rate:      {invalid_rate:.4f} ({invalid_rate * 100:.2f}%)")

# ============================================================
# WANDB LOGGING
# ============================================================

wandb.log({
    "benchmark_size": total,
    "correct": correct,
    "incorrect": total - correct - invalid,
    "invalid": invalid,
    "accuracy": accuracy,
    "invalid_rate": invalid_rate,
})

# ============================================================
# SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_path = "base_benchmark_results.csv"

results_df.to_csv(
    results_path,
    index=False,
)

print(f"\nResults saved to: {results_path}")

# Log prediction table to W&B
wandb_table = wandb.Table(
    dataframe=results_df
)

wandb.log({
    "benchmark_predictions": wandb_table
})

wandb.finish()

README.md:   0%|          | 0.00/301 [00:00<?, ?B/s]

benchmark_300_hard.csv:   0%|          | 0.00/110k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating benchmark split: 0 examples [00:00, ? examples/s]

Generating benchmark_hard split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 100
    })
    benchmark: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 80
    })
    benchmark_hard: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 300
    })
})
Benchmark examples: 300

Columns:
['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text']

First example:
{'id': 'benchmark_0001', 'crop': 'maize', 'category': 'nutrient', 'condition': 'Nitrogen deficiency', 'question': 'Crop: Maize\nField observations: older leaves show yellow-green V-shaped discoloration beginning near the tip and extending toward the midr

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]


Starting benchmark...

[001/300] Expected=B Predicted=D Correct=False
[002/300] Expected=D Predicted=B Correct=False
[003/300] Expected=A Predicted=B Correct=False
[004/300] Expected=C Predicted=A Correct=False
[005/300] Expected=A Predicted=A Correct=True
[006/300] Expected=C Predicted=B Correct=False
[007/300] Expected=A Predicted=A Correct=True
[008/300] Expected=C Predicted=A Correct=False
[009/300] Expected=B Predicted=B Correct=True
[010/300] Expected=B Predicted=B Correct=True
[011/300] Expected=D Predicted=B Correct=False
[012/300] Expected=D Predicted=B Correct=False
[013/300] Expected=A Predicted=D Correct=False
[014/300] Expected=B Predicted=B Correct=True
[015/300] Expected=A Predicted=A Correct=True
[016/300] Expected=C Predicted=C Correct=True
[017/300] Expected=A Predicted=A Correct=True
[018/300] Expected=B Predicted=A Correct=False
[019/300] Expected=C Predicted=A Correct=False
[020/300] Expected=A Predicted=A Correct=True
[021/300] Expected=D Predicted=C Correct=Fals

accuracy,▁
benchmark_size,▁
correct,▁
incorrect,▁
invalid,▁
invalid_rate,▁
accuracy,0.72333
benchmark_size,300
correct,217
incorrect,83
invalid,0


In [ ]:
import os
import re
import json
import torch
import pandas as pd
import wandb
from peft import PeftModel
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

# ============================================================
# CONFIG
# ============================================================

# HF_TOKEN = os.environ["HF_TOKEN"]
# WANDB_PROJECT = "agriculture-finetuning"

# bch_dataset_id = "DebasishDhal99/agriculture-synthetic-finetuning-dataset"
# ft_dataset_id = "DebasishDhal99/agriculture-synthetic-finetuning-dataset"

# base_model_id = "meta-llama/Llama-3.2-3B-Instruct"
# model_name = base_model_id.split("/")[-1]

DATASET_CONFIG = "v0"
BENCHMARK_SPLIT = "benchmark_hard"

MAX_NEW_TOKENS = 8

# ============================================================
# WANDB
# ============================================================

wandb.init(
    project=WANDB_PROJECT_NAME,
    name=f"lora-previous-benchmark-hard_synthetic-data-{model_name}",
    config={
        "model": base_model_id,
        "dataset": bch_dataset_id,
        "dataset_config": DATASET_CONFIG,
        "benchmark_split": BENCHMARK_SPLIT,
        "max_new_tokens": MAX_NEW_TOKENS,
    },
)

ADAPTER_ID = "DebasishDhal99/Llama-3.2-3B-Instruct-agriculture-lora"

# ============================================================
# LOAD DATASET
# ============================================================

dataset = load_dataset(
    bch_dataset_id,
    name=DATASET_CONFIG,
    token=hf_token,
)

benchmark = dataset[BENCHMARK_SPLIT]

print(dataset)
print(f"Benchmark examples: {len(benchmark)}")

# ============================================================
# INSPECT COLUMNS
# ============================================================

print("\nColumns:")
print(benchmark.column_names)

print("\nFirst example:")
print(benchmark[0])

# ============================================================
# LOAD MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    token=hf_token,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    token=hf_token,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    model,
    ADAPTER_ID,
    token=hf_token,
)

model.eval()

# Llama doesn't normally have a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ============================================================
# HELPERS
# ============================================================

def get_question(row):
    """
    Supports the CSV structure generated previously:

        question
        answer
        answer_text

    and also ChatML-style messages if present.
    """

    # Case 1: explicit question column
    if "question" in row:
        return row["question"]

    # Case 2: messages column
    if "messages" in row:
        messages = row["messages"]

        if isinstance(messages, str):
            messages = json.loads(messages)

        for message in messages:
            if message["role"] == "user":
                return message["content"]

    raise ValueError(
        f"Cannot find question column. Available columns: {row.keys()}"
    )


def get_expected_answer(row):
    """
    Expected answer should be A/B/C/D.
    """

    answer = row.get("answer", None)

    if answer is None:
        raise ValueError(
            f"Cannot find answer column. Available columns: {row.keys()}"
        )

    answer = str(answer).strip().upper()

    # Extract A/B/C/D if answer contains additional text
    match = re.search(r"\b([ABCD])\b", answer)

    if match:
        return match.group(1)

    raise ValueError(f"Could not parse expected answer: {answer}")


def extract_prediction(text):
    """
    Extract A/B/C/D from model output.

    We strongly prefer the first standalone A/B/C/D.
    """

    text = text.strip().upper()

    # First try a standalone letter
    match = re.search(r"\b([ABCD])\b", text)

    if match:
        return match.group(1)

    # Handle outputs such as:
    # "A."
    # "(A)"
    # "Answer: A"
    match = re.search(r"(?:^|[\s:(])([ABCD])(?:[.)\s]|$)", text)

    if match:
        return match.group(1)

    return None


# ============================================================
# EVALUATION
# ============================================================

results = []

correct = 0
invalid = 0

print("\nStarting benchmark...\n")

for idx, row in enumerate(benchmark):

    question = get_question(row)
    expected = get_expected_answer(row)

    messages = [
        {
            "role": "system",
            "content": (
                "You are an agricultural question answering assistant. "
                "Answer the multiple-choice question by returning only "
                "the letter A, B, C, or D."
            ),
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    prediction = extract_prediction(generated_text)

    is_correct = prediction == expected

    if prediction is None:
        invalid += 1
    elif is_correct:
        correct += 1

    results.append({
        "index": idx,
        "question": question,
        "expected": expected,
        "prediction": prediction,
        "raw_output": generated_text,
        "correct": is_correct,
    })

    print(
        f"[{idx + 1:03d}/{len(benchmark)}] "
        f"Expected={expected} "
        f"Predicted={prediction} "
        f"Correct={is_correct}"
    )


# ============================================================
# METRICS
# ============================================================

total = len(results)

accuracy = correct / total
invalid_rate = invalid / total

print("\n" + "=" * 60)
print("BASE MODEL BENCHMARK")
print("=" * 60)

print(f"Model:             {base_model_id}")
print(f"Benchmark size:    {total}")
print(f"Correct:           {correct}")
print(f"Incorrect:         {total - correct - invalid}")
print(f"Invalid:           {invalid}")
print(f"Accuracy:          {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Invalid rate:      {invalid_rate:.4f} ({invalid_rate * 100:.2f}%)")

# ============================================================
# WANDB LOGGING
# ============================================================

wandb.log({
    "benchmark_size": total,
    "correct": correct,
    "incorrect": total - correct - invalid,
    "invalid": invalid,
    "accuracy": accuracy,
    "invalid_rate": invalid_rate,
})

# ============================================================
# SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_path = "base_benchmark_results.csv"

results_df.to_csv(
    results_path,
    index=False,
)

print(f"\nResults saved to: {results_path}")

# Log prediction table to W&B
wandb_table = wandb.Table(
    dataframe=results_df
)

wandb.log({
    "benchmark_predictions": wandb_table
})

wandb.finish()

train/entropy,▁
train/epoch,▁
train/global_step,▁
train/grad_norm,▁
train/learning_rate,▁
train/loss,▁
train/mean_token_accuracy,▁
train/num_tokens,▁
train/entropy,1.81693
train/epoch,0.16
train/global_step,10


DatasetDict({
    train: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 100
    })
    benchmark: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 80
    })
    benchmark_hard: Dataset({
        features: ['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text'],
        num_rows: 300
    })
})
Benchmark examples: 300

Columns:
['id', 'crop', 'category', 'condition', 'question', 'answer', 'answer_text']

First example:
{'id': 'benchmark_0001', 'crop': 'maize', 'category': 'nutrient', 'condition': 'Nitrogen deficiency', 'question': 'Crop: Maize\nField observations: older leaves show yellow-green V-shaped discoloration beginning near the tip and extending toward the midr

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 97.3MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            


Starting benchmark...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[001/300] Expected=B Predicted=A Correct=False
[002/300] Expected=D Predicted=B Correct=False
[003/300] Expected=A Predicted=B Correct=False
[004/300] Expected=C Predicted=B Correct=False
[005/300] Expected=A Predicted=A Correct=True
[006/300] Expected=C Predicted=B Correct=False
[007/300] Expected=A Predicted=B Correct=False
[008/300] Expected=C Predicted=A Correct=False
[009/300] Expected=B Predicted=B Correct=True
[010/300] Expected=B Predicted=B Correct=True
[011/300] Expected=D Predicted=B Correct=False
[012/300] Expected=D Predicted=D Correct=True
[013/300] Expected=A Predicted=D Correct=False
[014/300] Expected=B Predicted=B Correct=True
[015/300] Expected=A Predicted=A Correct=True
[016/300] Expected=C Predicted=C Correct=True
[017/300] Expected=A Predicted=A Correct=True
[018/300] Expected=B Predicted=A Correct=False
[019/300] Expected=C Predicted=C Correct=True
[020/300] Expected=A Predicted=A Correct=True
[021/300] Expected=D Predicted=None Correct=False
[022/300] Expected=A

accuracy,▁
benchmark_size,▁
correct,▁
incorrect,▁
invalid,▁
invalid_rate,▁
accuracy,0.71
benchmark_size,300
correct,213
incorrect,76
invalid,11


### Fine-tuning with synthetic data

In [ ]:
!pip install --force-reinstall numpy

  Using cached numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.3
    Uninstalling numpy-2.5.3:
      Successfully uninstalled numpy-2.5.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.
bigframes 2.49.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 

In [ ]:
# ============================================================
# LoRA SFT — Agriculture
#
# Base model + synthetic agricultural dataset
# NO QUANTIZATION
# Adapter-only output
# W&B + Hugging Face Hub
#
# Dataset:
#   train       -> 1000
#   validation  -> 100
#   benchmark   -> 300/80, NEVER USED HERE
# ============================================================

import os
import re
import logging
import time
import platform

import torch
import wandb

from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig


# ============================================================
# CONFIG
# ============================================================

HF_TOKEN = hf_token
WANDB_API_KEY = os.environ["WANDB_API_KEY"]

MODEL_NAME = base_model_id

TRAIN_DATASET = ft_dataset_id
DATASET_CONFIG = "v0"

HF_REPO_ID = (
    f"DebasishDhal99/{model_name}-synthetic-agriculture-lora"
)

WANDB_PROJECT = WANDB_PROJECT_NAME

RUN_NAME = f"{model_name}-synthetic-agriculture-lora"

OUTPUT_DIR = (
    f"./{model_name}-synthetic-agriculture-lora"
)

SEED = 42

EPOCHS = 2
LEARNING_RATE = 2e-4

BATCH_SIZE = 2
GRAD_ACCUMULATION = 8

MAX_LENGTH = 1024

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# How often to run validation.
#
# Your effective batch size is:
#
# 2 * 8 = 16
#
# 1000 / 16 ~= 63 optimizer steps / epoch
#
# Therefore 25 steps gives several validation points
# during a 2-epoch run.
EVAL_STEPS = 25

# Generation settings for MCQ validation
MCQ_MAX_NEW_TOKENS = 8


# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)


# ============================================================
# DTYPE
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU required for this training setup."
    )

USE_BF16 = torch.cuda.is_bf16_supported()

DTYPE = (
    torch.bfloat16
    if USE_BF16
    else torch.float16
)

logger.info("Using dtype: %s", DTYPE)


# ============================================================
# W&B
# ============================================================

wandb.login(key=WANDB_API_KEY)

wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,

    config={

        # ----------------------------------------------------
        # MODEL
        # ----------------------------------------------------

        "base_model": MODEL_NAME,
        "model_name": model_name,

        # ----------------------------------------------------
        # TRAINING METHOD
        # ----------------------------------------------------

        "training_method": "LoRA",
        "quantization": "none",

        # ----------------------------------------------------
        # DATASET
        # ----------------------------------------------------

        "dataset": TRAIN_DATASET,
        "dataset_config": DATASET_CONFIG,

        "train_split": "train",
        "validation_split": "validation",
        "benchmark_split": "benchmark",

        "train_examples": 1000,
        "validation_examples": 100,

        # Benchmark is intentionally NOT used during training
        "benchmark_used_during_training": False,

        # ----------------------------------------------------
        # TRAINING
        # ----------------------------------------------------

        "epochs": EPOCHS,

        "learning_rate": LEARNING_RATE,

        "optimizer": "adamw_torch",

        "weight_decay": 0.01,

        "lr_scheduler": "linear",

        "warmup_ratio": 0.0,

        # ----------------------------------------------------
        # BATCHING
        # ----------------------------------------------------

        "per_device_train_batch_size": BATCH_SIZE,

        "per_device_eval_batch_size": BATCH_SIZE,

        "gradient_accumulation_steps":
            GRAD_ACCUMULATION,

        "effective_batch_size":
            BATCH_SIZE * GRAD_ACCUMULATION,

        # ----------------------------------------------------
        # SEQUENCE
        # ----------------------------------------------------

        "max_length": MAX_LENGTH,

        # ----------------------------------------------------
        # LoRA
        # ----------------------------------------------------

        "lora_r": LORA_R,

        "lora_alpha": LORA_ALPHA,

        "lora_dropout": LORA_DROPOUT,

        "lora_bias": "none",

        "lora_task_type": "CAUSAL_LM",

        "lora_target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],

        # ----------------------------------------------------
        # PRECISION
        # ----------------------------------------------------

        "dtype": str(DTYPE),

        "bf16": USE_BF16,

        "fp16": not USE_BF16,

        "gradient_checkpointing": False,

        # ----------------------------------------------------
        # EVALUATION
        # ----------------------------------------------------

        "eval_strategy": "steps",

        "eval_steps": EVAL_STEPS,

        "evaluation_metric": "MCQ accuracy",

        "selection_metric": "eval_mcq_accuracy",

        "greater_is_better": True,

        "eval_loss_logged": True,

        "mcq_accuracy_logged": True,

        "invalid_answer_rate_logged": True,

        # ----------------------------------------------------
        # CHECKPOINTING
        # ----------------------------------------------------

        "save_strategy": "steps",

        "save_steps": EVAL_STEPS,

        "save_total_limit": 3,

        "load_best_model_at_end": True,

        # ----------------------------------------------------
        # GENERATION / MCQ EVALUATION
        # ----------------------------------------------------

        "mcq_max_new_tokens":
            MCQ_MAX_NEW_TOKENS,

        "mcq_decoding": "greedy",

        "mcq_temperature": None,

        # ----------------------------------------------------
        # REPRODUCIBILITY
        # ----------------------------------------------------

        "seed": SEED,

        "data_seed": SEED,

        # ----------------------------------------------------
        # HARDWARE / SOFTWARE
        # ----------------------------------------------------

        "python": platform.python_version(),

        "torch": torch.__version__,

        "cuda": torch.version.cuda,

        "gpu": torch.cuda.get_device_name(0),

        # ----------------------------------------------------
        # OUTPUT
        # ----------------------------------------------------

        "output_dir": OUTPUT_DIR,

        "hf_repo": HF_REPO_ID,

        "adapter_only": True,
    },
)

# ============================================================
# LOAD DATASET
# ============================================================

logger.info("Loading Hugging Face dataset...")

train_raw = load_dataset(
    TRAIN_DATASET,
    name=DATASET_CONFIG,
    split="train",
    token=HF_TOKEN,
)

validation_raw = load_dataset(
    TRAIN_DATASET,
    name=DATASET_CONFIG,
    split="validation",
    token=HF_TOKEN,
)

logger.info(
    "Raw train examples: %d",
    len(train_raw),
)

logger.info(
    "Raw validation examples: %d",
    len(validation_raw),
)

wandb.config.update({
    "raw_train_examples": len(train_raw),
    "raw_validation_examples": len(validation_raw),
})


# ============================================================
# DATA CLEANING
# ============================================================

def prepare_dataframe(dataset, split_name):

    df = dataset.to_pandas()

    logger.info(
        "%s columns: %s",
        split_name,
        list(df.columns),
    )

    # --------------------------------------------------------
    # Question
    # --------------------------------------------------------

    if "question" not in df.columns:
        raise ValueError(
            f"{split_name} dataset has no 'question' column. "
            f"Columns: {list(df.columns)}"
        )

    df["question"] = (
        df["question"]
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------------
    # Answer
    #
    # Your previous dataset used "answers".
    # The newer generated benchmark uses "answer".
    #
    # Support both.
    # --------------------------------------------------------

    if "answers" in df.columns:
        answer_column = "answers"

    elif "answer" in df.columns:
        answer_column = "answer"

    else:
        raise ValueError(
            f"{split_name} dataset has neither "
            "'answers' nor 'answer'. "
            f"Columns: {list(df.columns)}"
        )

    df[answer_column] = (
        df[answer_column]
        .astype(str)
        .str.strip()
    )

    # Standardize to "answer"
    if answer_column != "answer":
        df["answer"] = df[answer_column]

    # --------------------------------------------------------
    # Filter invalid rows
    # --------------------------------------------------------

    before = len(df)

    df = df[
        (df["question"].str.len() > 5)
        &
        (df["answer"].str.len() > 0)
    ]

    df = df.drop_duplicates(
        subset=["question"]
    ).reset_index(drop=True)

    after = len(df)

    logger.info(
        "%s: %d -> %d examples",
        split_name,
        before,
        after,
    )

    return df


train_df = prepare_dataframe(
    train_raw,
    "train",
)

validation_df = prepare_dataframe(
    validation_raw,
    "validation",
)


wandb.config.update({
    "clean_train_examples": len(train_df),
    "clean_validation_examples": len(validation_df),
})


# ============================================================
# CHAT FORMAT
# ============================================================

def format_example(example):

    return {
        "messages": [
            {
                "role": "user",
                "content": example["question"],
            },
            {
                "role": "assistant",
                "content": example["answer"],
            },
        ]
    }


train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False,
)

validation_dataset = Dataset.from_pandas(
    validation_df,
    preserve_index=False,
)


train_dataset = train_dataset.map(
    format_example,
    remove_columns=train_dataset.column_names,
)

validation_dataset = validation_dataset.map(
    format_example,
    remove_columns=validation_dataset.column_names,
)


logger.info(
    "Train: %d | Validation: %d",
    len(train_dataset),
    len(validation_dataset),
)


# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# MODEL
# ============================================================

logger.info(
    "Loading base model WITHOUT quantization..."
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=DTYPE,
    # device_map="auto",
    trust_remote_code=True,
)

model = model.to("cuda:0")

model.config.use_cache = False


# ============================================================
# LoRA CONFIG
# ============================================================

peft_config = LoraConfig(
    r=LORA_R,

    lora_alpha=LORA_ALPHA,

    lora_dropout=LORA_DROPOUT,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",

    task_type="CAUSAL_LM",
)


# ============================================================
# MCQ VALIDATION
# ============================================================

def extract_answer(text):

    """
    Extract A/B/C/D from model output.

    Handles outputs such as:

        A
        A.
        (A)
        Answer: A
        A. Nitrogen deficiency
    """

    text = text.strip().upper()

    # Prefer explicit answer patterns
    patterns = [
        r"ANSWER\s*[:\-]?\s*([ABCD])\b",
        r"^\s*\(?([ABCD])\)?(?:[\.\):\s]|$)",
        r"\b([ABCD])\b",
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
        )

        if match:
            return match.group(1)

    return None


def normalize_expected_answer(answer):

    answer = str(answer).strip().upper()

    match = re.search(
        r"\b([ABCD])\b",
        answer,
    )

    if match:
        return match.group(1)

    return None


def evaluate_mcq_accuracy(
    model,
    dataset,
    tokenizer,
):

    was_training = model.training

    model.eval()

    correct = 0
    invalid = 0
    total = len(dataset)

    predictions = []

    for row in dataset:

        messages = row["messages"]

        # ----------------------------------------------------
        # Convert chat messages into Llama prompt
        # ----------------------------------------------------

        prompt = tokenizer.apply_chat_template(
            messages[:1],
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
        )

        inputs = {
            k: v.to(model.device)
            for k, v in inputs.items()
        }

        # ----------------------------------------------------
        # Generate answer
        # ----------------------------------------------------

        with torch.no_grad():

            outputs = model.generate(
                **inputs,

                max_new_tokens=MCQ_MAX_NEW_TOKENS,

                do_sample=False,

                pad_token_id=tokenizer.eos_token_id,
            )

        generated_tokens = outputs[
            0
        ][
            inputs["input_ids"].shape[1]:
        ]

        generated_text = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        ).strip()

        prediction = extract_answer(
            generated_text
        )

        # ----------------------------------------------------
        # Expected answer
        # ----------------------------------------------------

        expected = normalize_expected_answer(
            messages[1]["content"]
        )

        is_correct = (
            prediction is not None
            and prediction == expected
        )

        if prediction is None:

            invalid += 1

        elif is_correct:

            correct += 1

        predictions.append({
            "expected": expected,
            "prediction": prediction,
            "raw_output": generated_text,
            "correct": is_correct,
        })

    accuracy = (
        correct / total
        if total > 0
        else 0.0
    )

    invalid_rate = (
        invalid / total
        if total > 0
        else 0.0
    )

    if was_training:
        model.train()

    return {
        "accuracy": accuracy,
        "invalid_rate": invalid_rate,
        "correct": correct,
        "invalid": invalid,
        "total": total,
        "predictions": predictions,
    }


# ============================================================
# CUSTOM TRAINER
#
# Adds actual MCQ accuracy to the normal eval metrics.
# ============================================================

class MCQSFTTrainer(SFTTrainer):

    def evaluate(
        self,
        eval_dataset=None,
        ignore_keys=None,
        metric_key_prefix="eval",
    ):

        # ----------------------------------------------------
        # Normal HF/TRL evaluation
        #
        # Gives us eval_loss.
        # ----------------------------------------------------

        metrics = super().evaluate(
            eval_dataset=eval_dataset,
            ignore_keys=ignore_keys,
            metric_key_prefix=metric_key_prefix,
        )

        dataset_to_use = (
            eval_dataset
            if eval_dataset is not None
            else self.eval_dataset
        )

        # ----------------------------------------------------
        # Actual MCQ accuracy
        # ----------------------------------------------------

        mcq_results = evaluate_mcq_accuracy(
            self.model,
            dataset_to_use,
            self.processing_class,
        )

        accuracy_key = (
            f"{metric_key_prefix}_mcq_accuracy"
        )

        invalid_key = (
            f"{metric_key_prefix}_invalid_rate"
        )

        metrics[accuracy_key] = (
            mcq_results["accuracy"]
        )

        metrics[invalid_key] = (
            mcq_results["invalid_rate"]
        )

        # ----------------------------------------------------
        # Log to W&B
        # ----------------------------------------------------

        self.log({
            accuracy_key: mcq_results["accuracy"],
            invalid_key: mcq_results["invalid_rate"],
        })

        logger.info(
            "MCQ validation accuracy: %.2f%% "
            "(%d/%d)",
            mcq_results["accuracy"] * 100,
            mcq_results["correct"],
            mcq_results["total"],
        )

        return metrics


# ============================================================
# TRAINING CONFIG
# ============================================================

training_args = SFTConfig(

    output_dir=OUTPUT_DIR,

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,

    per_device_eval_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUMULATION,

    learning_rate=LEARNING_RATE,

    max_length=MAX_LENGTH,

    weight_decay=0.01,

    loss_type="nll",

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    eval_strategy="steps",

    eval_steps=EVAL_STEPS,

    # --------------------------------------------------------
    # Checkpoints
    # --------------------------------------------------------

    save_strategy="steps",

    save_steps=EVAL_STEPS,

    save_total_limit=3,

    # --------------------------------------------------------
    # IMPORTANT
    #
    # Select best checkpoint by actual agricultural MCQ
    # validation accuracy, NOT eval_loss.
    # --------------------------------------------------------

    load_best_model_at_end=True,

    metric_for_best_model="eval_mcq_accuracy",

    greater_is_better=True,

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    logging_strategy="steps",

    logging_steps=10,

    # --------------------------------------------------------
    # Precision
    # --------------------------------------------------------

    bf16=USE_BF16,

    fp16=not USE_BF16,

    gradient_checkpointing=False,

    # --------------------------------------------------------
    # W&B
    # --------------------------------------------------------

    report_to="wandb",

    run_name=RUN_NAME,

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    seed=SEED,

    data_seed=SEED,
)


# ============================================================
# TRAINER
# ============================================================

trainer = MCQSFTTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=validation_dataset,

    processing_class=tokenizer,

    peft_config=peft_config,
)


# ============================================================
# VERIFY TRAINABLE PARAMETERS
# ============================================================

trainable_params = sum(
    p.numel()
    for p in trainer.model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in trainer.model.parameters()
)

trainable_percentage = (
    100 * trainable_params / total_params
)

logger.info(
    "Total parameters: %.2f B",
    total_params / 1e9,
)

logger.info(
    "Trainable parameters: %.2f M",
    trainable_params / 1e6,
)

logger.info(
    "Trainable percentage: %.4f%%",
    trainable_percentage,
)

wandb.config.update({
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "trainable_percentage": trainable_percentage,
})


# ============================================================
# TRAIN
# ============================================================

logger.info(
    "Starting LoRA training..."
)

start_time = time.time()

train_result = trainer.train()

training_time = (
    time.time() - start_time
)

logger.info(
    "Training finished in %.2f minutes",
    training_time / 60,
)

wandb.log({
    "training/time_minutes":
        training_time / 60,
})


# ============================================================
# SAVE BEST ADAPTER
# ============================================================

logger.info(
    "Saving best LoRA adapter to %s",
    OUTPUT_DIR,
)

trainer.save_model(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)


# ============================================================
# FINAL VALIDATION
# ============================================================

logger.info(
    "Running final validation..."
)

final_eval = trainer.evaluate()

logger.info(
    "Final validation metrics:"
)

for key, value in final_eval.items():

    logger.info(
        "  %s = %s",
        key,
        value,
    )

    if isinstance(value, (int, float)):

        wandb.log({
            f"final_eval/{key}": value
        })


# ============================================================
# PERPLEXITY
# ============================================================

if "eval_loss" in final_eval:

    perplexity = torch.exp(
        torch.tensor(
            final_eval["eval_loss"]
        )
    ).item()

    logger.info(
        "Validation perplexity: %.4f",
        perplexity,
    )

    wandb.log({
        "final_eval/perplexity":
            perplexity
    })


# ============================================================
# W&B ADAPTER ARTIFACT
# ============================================================

artifact = wandb.Artifact(
    name=f"{RUN_NAME}-adapter",
    type="model",

    metadata={

        "base_model": MODEL_NAME,

        "dataset": TRAIN_DATASET,

        "dataset_config": DATASET_CONFIG,

        "training_method": "LoRA",

        "quantization": "none",

        "dtype": str(DTYPE),

        "epochs": EPOCHS,

        "learning_rate": LEARNING_RATE,

        "batch_size": BATCH_SIZE,

        "gradient_accumulation":
            GRAD_ACCUMULATION,

        "lora_r": LORA_R,

        "lora_alpha": LORA_ALPHA,

        "lora_dropout": LORA_DROPOUT,

        "max_length": MAX_LENGTH,

        "seed": SEED,

        "selection_metric":
            "eval_mcq_accuracy",
    },
)

artifact.add_dir(
    OUTPUT_DIR
)

wandb.log_artifact(
    artifact
)


# ============================================================
# HUGGING FACE — ADAPTER ONLY
# ============================================================

logger.info(
    "Uploading LoRA adapter to %s",
    HF_REPO_ID,
)


README_CONTENT = f"""---
base_model: {MODEL_NAME}
library_name: peft
pipeline_tag: text-generation
tags:
  - agriculture
  - agricultural-qa
  - lora
  - peft
  - llama
  - synthetic-data
  - instruction-tuning
---

# {model_name} — Synthetic Agriculture LoRA

LoRA adapter fine-tuned from
[`{MODEL_NAME}`](https://huggingface.co/{MODEL_NAME})
on a synthetic agricultural question-answering dataset.

This repository contains the **LoRA adapter only**, not a merged
copy of the base model.

## Model

| Property | Value |
|---|---|
| Base model | `{MODEL_NAME}` |
| Fine-tuning method | LoRA |
| Quantization | None |
| Task | Agricultural instruction following / MCQ |
| Adapter only | Yes |

## Dataset

Dataset:

[`{TRAIN_DATASET}`](https://huggingface.co/datasets/{TRAIN_DATASET})

Configuration:

`{DATASET_CONFIG}`

| Split | Examples | Purpose |
|---|---:|---|
| train | 1,000 | LoRA training |
| validation | 100 | Checkpoint selection / validation |
| benchmark | 300 | Final evaluation only |

The benchmark split is **not used during fine-tuning or
checkpoint selection**.

## Training Configuration

### Optimization

| Parameter | Value |
|---|---|
| Epochs | `{EPOCHS}` |
| Learning rate | `{LEARNING_RATE}` |
| Optimizer | AdamW |
| Weight decay | `0.01` |
| LR scheduler | Linear |
| Warmup ratio | `0.0` |

### Batch / sequence

| Parameter | Value |
|---|---|
| Per-device train batch size | `{BATCH_SIZE}` |
| Per-device eval batch size | `{BATCH_SIZE}` |
| Gradient accumulation | `{GRAD_ACCUMULATION}` |
| Effective batch size | `{BATCH_SIZE * GRAD_ACCUMULATION}` |
| Maximum sequence length | `{MAX_LENGTH}` |

### LoRA

| Parameter | Value |
|---|---|
| Rank (`r`) | `{LORA_R}` |
| Alpha | `{LORA_ALPHA}` |
| Dropout | `{LORA_DROPOUT}` |
| Bias | `none` |
| Task type | `CAUSAL_LM` |

Target modules:

```text
q_proj
k_proj
v_proj
o_proj
gate_proj
up_proj
down_proj
"""

README_PATH = os.path.join(OUTPUT_DIR, "README.md")

with open(README_PATH, "w", encoding="utf-8") as f:
    f.write(README_CONTENT)

from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True,
)

api.upload_file(
    path_or_fileobj=README_PATH,
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model",
)

trainer.model.push_to_hub(
    HF_REPO_ID,
    token=HF_TOKEN,
)

tokenizer.push_to_hub(
    HF_REPO_ID,
    token=HF_TOKEN,
)

logger.info(
    "LoRA adapter uploaded successfully."
)


# ============================================================
# FINISH
# ============================================================

wandb.finish()

logger.info(
    "Experiment complete."
)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: debasishdhal to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


README.md:   0%|          | 0.00/301 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/321k [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/33.3k [00:00<?, ?B/s]

benchmark.csv:   0%|          | 0.00/24.5k [00:00<?, ?B/s]

...

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
25,0.790023,1.039449,0.777458,0.804400,36495.000000
50,0.216980,0.775210,0.368353,0.862410,73090.000000
75,0.156119,0.730483,0.308930,0.871101,109109.000000
100,0.130915,0.761007,0.274590,0.875358,145664.000000
125,0.125270,0.769897,0.259018,0.873834,182327.000000
126,0.125270,0.769778,0.258346,0.873881,183040.000000


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
/usr/local/lib/python3.13/dist-packages/peft/utils/other.py:1508: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6ac67d66-530fe6da4888d2b80501b645;7a42b947-af12-4d8c-a997-c30f443e4cb5)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-3B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.2-

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
0.125270,0.775210,126,0.368353,0.862410,183040.000000


wandb: Adding directory to artifact (Llama-3.2-3B-Instruct-synthetic-agriculture-lora)... Done. 79.7s


RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-6ac6843f-7f8f056b754306276904b5e4;ffece293-5b12-4481-bdc2-eeaf4d8fe4ff)

Repository Not Found for url: https://huggingface.co/api/models/DebasishDhal99/Llama-3.2-3B-Instruct-synthetic-agriculture-lora/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.

In [ ]:
print(HF_REPO_ID)

DebasishDhal99/Llama-3.2-3B-Instruct-synthetic-agriculture-lora


In [ ]:
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True,
)

api.upload_file(
    path_or_fileobj=README_PATH,
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model",
)

trainer.model.push_to_hub(
    HF_REPO_ID,
    token=HF_TOKEN,
)

tokenizer.push_to_hub(
    HF_REPO_ID,
    token=HF_TOKEN,
)


README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/utils/other.py:1508: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6ac68543-2c2704b849e111284913acbc;9bc03cd0-2af6-4e4d-bb34-6c92d29242b7)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-3B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.2-3B-Instruct.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:313: UserWarning: Could not find a config file in meta-llama/Llama-3.2-3B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.8kB / 97.3MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpww2ncvv8/tokenizer.json:  92%|#########2| 15.9MB / 17.2MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/DebasishDhal99/Llama-3.2-3B-Instruct-synthetic-agriculture-lora/commit/b543f90fe26300ef1311650a52e801feb076960f', commit_message='Upload tokenizer', commit_description='', oid='b543f90fe26300ef1311650a52e801feb076960f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/DebasishDhal99/Llama-3.2-3B-Instruct-synthetic-agriculture-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='DebasishDhal99/Llama-3.2-3B-Instruct-synthetic-agriculture-lora'), pr_revision=None, pr_num=None)